## Initialisation

Ce notebook construit la couche **gold** à partir des données silver.

On commence par établir la connexion à la base PostgreSQL, puis on charge les objets silver depuis les 4 tables sources.

In [1]:
from sqlalchemy import create_engine, text, URL

DB_USER = "indusense_user"
DB_PASSWORD = "ThEP@ssW0rd"
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "indusense_db"

url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)

engine = create_engine(url)

try:
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print(f"✅ Connexion à PostgreSQL réussie ({DB_HOST}:{DB_PORT}/{DB_NAME})")
except Exception as e:
    print(f"❌ Échec de connexion : {e}")

✅ Connexion à PostgreSQL réussie (localhost:5432/indusense_db)


In [2]:
from sqlalchemy.orm import Session
from sqlalchemy import select
from models.silver import SilverRelevesIncidents, SilverTelemetry, SilverMachine, SilverMaintenance

with Session(engine) as session:
    silver_incidents = session.execute(select(SilverRelevesIncidents)).scalars().all()
    silver_telemetry = session.execute(select(SilverTelemetry)).scalars().all()
    silver_machines = session.execute(select(SilverMachine)).scalars().all()
    silver_maintenances = session.execute(select(SilverMaintenance)).scalars().all()

    session.expunge_all()

print(f"✅ silver_releves_incidents : {len(silver_incidents)} lignes chargées")
print(f"✅ silver_telemetry         : {len(silver_telemetry)} lignes chargées")
print(f"✅ silver_machine           : {len(silver_machines)} lignes chargées")
print(f"✅ silver_maintenance       : {len(silver_maintenances)} lignes chargées")

✅ silver_releves_incidents : 3735 lignes chargées
✅ silver_telemetry         : 400089 lignes chargées
✅ silver_machine           : 45 lignes chargées
✅ silver_maintenance       : 4686 lignes chargées


## Feature engineering

Construction des objets `GoldTelemetry`. Pour chaque relevé, on calcule l'historique glissant de la machine sur les fenêtres 6h/12h/24h précédant la mesure, pour chaque métrique (`temperature`, `pressure`, `voltage`, `rotation`) et chaque statistique (`mean`, `max`, `std`).

On ajoute aussi, pour la température uniquement, des variables de tendance sur 1h, 3h et 6h : `temperature_trend_Xh = temperature_c(t) - temperature_c(t - Xh)`, pour détecter la direction et la vitesse du changement.

Un z-score par machine, pour chaque type de métrique : `{type}_zscore_machine = (valeur - moyenne_machine) / écart_type_machine`. ⚠️ La baseline (moyenne/écart-type) est calculée sur tout l'historique de la machine faute de split train/val/test — à recalculer sur le train uniquement une fois l'étape 8 (split temporel) ajoutée, pour éviter le data leakage.

Enfin, le **label** panne sur 4 horizons (`label_failure_next_6h/12h/24h/48h`) : pour chaque relevé, vrai si un incident survient pour cette machine dans les X heures qui suivent. Calculé avec le principe du roadmap (rolling en arrière = on inverse l'axe du temps, on fait un rolling classique — donc vers l'arrière sur l'axe inversé —, puis on ré-inverse), adapté ici à un historique irrégulier (pas de grille horaire fixe) via une timeline combinée télémétrie + incidents.

In [ ]:
import pandas as pd
from models.gold import GoldTelemetry

METRICS = {
    "temperature": "temperature_c",
    "pressure": "pressure_bar",
    "voltage": "voltage_mean_v",
    "rotation": "rotation_mean_rpm",
}
INTERVALS = ["6h", "12h", "24h"]
STATS = ["mean", "max", "std"]
TREND_INTERVALS = {"1h": pd.Timedelta(hours=1), "3h": pd.Timedelta(hours=3), "6h": pd.Timedelta(hours=6)}
LABEL_INTERVALS = ["6h", "12h", "24h", "48h"]

df = pd.DataFrame([{
    "machine_id": row.machine_id,
    "date": row.date,
    "temperature_c": row.temperature_c,
    "pressure_bar": row.pressure_bar,
    "voltage_mean_v": row.voltage_mean_v,
    "rotation_mean_rpm": row.rotation_mean_rpm,
    "pieces_produced": row.pieces_produced,
} for row in silver_telemetry]).sort_values(["machine_id", "date"]).reset_index(drop=True)

FEATURE_COLUMNS = []
for metric_name, column in METRICS.items():
    for interval in INTERVALS:
        rolling = df.set_index("date").groupby("machine_id")[column].rolling(interval)
        stats_df = rolling.agg(["mean", "max", "std"]).reset_index(level=0, drop=True)
        for stat in STATS:
            feature_name = f"{metric_name}_{stat}_{interval}"
            df[feature_name] = stats_df[stat].values
            FEATURE_COLUMNS.append(feature_name)

temperature_history = df[["machine_id", "date", "temperature_c"]].rename(
    columns={"temperature_c": "temperature_past"}
).sort_values("date")

for label, delta in TREND_INTERVALS.items():
    query = df[["machine_id", "date"]].copy()
    query["target_date"] = query["date"] - delta
    query["row_id"] = df.index
    query = query.sort_values("target_date")

    matched = pd.merge_asof(
        query, temperature_history,
        left_on="target_date", right_on="date",
        by="machine_id", direction="backward", suffixes=("", "_history"),
    ).set_index("row_id")["temperature_past"].reindex(df.index)

    feature_name = f"temperature_trend_{label}"
    df[feature_name] = df["temperature_c"] - matched
    FEATURE_COLUMNS.append(feature_name)

# Baseline calculée sur tout l'historique de la machine (pas de split train/test pour l'instant → leakage à corriger plus tard)
for metric_name, column in METRICS.items():
    machine_stats = df.groupby("machine_id")[column].agg(["mean", "std"])
    machine_mean = df["machine_id"].map(machine_stats["mean"])
    machine_std = df["machine_id"].map(machine_stats["std"])

    feature_name = f"{metric_name}_zscore_machine"
    df[feature_name] = (df[column] - machine_mean) / machine_std
    FEATURE_COLUMNS.append(feature_name)

NUMERIC_COLUMNS = ["temperature_c", "pressure_bar", "voltage_mean_v", "rotation_mean_rpm"] + FEATURE_COLUMNS
df[NUMERIC_COLUMNS] = df[NUMERIC_COLUMNS].round(2)

# Label panne : rolling en arrière (reverse -> rolling trailing -> reverse) pour compter les
# incidents à venir dans une fenêtre future, sans hypothèse de grille horaire régulière.
incidents_df = pd.DataFrame([{
    "machine_id": row.machine_id,
    "date": row.date,
} for row in silver_incidents if row.machine_id is not None and row.date is not None])
incidents_df["incident_count"] = 1

timeline = pd.concat([
    df[["machine_id", "date"]].assign(incident_count=0, row_id=df.index),
    incidents_df.assign(row_id=-1),
], ignore_index=True)
timeline["reversed_date"] = timeline["date"].max() - timeline["date"]
timeline = timeline.sort_values(["machine_id", "reversed_date"]).reset_index(drop=True)

LABEL_COLUMNS = []
for interval in LABEL_INTERVALS:
    rolling = timeline.set_index("reversed_date").groupby("machine_id")["incident_count"].rolling(interval, min_periods=1)
    timeline["future_incident_count"] = rolling.sum().reset_index(level=0, drop=True).values

    label_name = f"label_failure_next_{interval}"
    future_counts = timeline[timeline["row_id"] >= 0].set_index("row_id")["future_incident_count"].reindex(df.index)
    df[label_name] = future_counts.fillna(0) > 0
    LABEL_COLUMNS.append(label_name)

gold_telemetry_objects = [GoldTelemetry(
    machine_id=row.machine_id,
    date=row.date,
    temperature_c=row.temperature_c,
    pressure_bar=row.pressure_bar,
    voltage_mean_v=row.voltage_mean_v,
    rotation_mean_rpm=row.rotation_mean_rpm,
    pieces_produced=int(row.pieces_produced) if pd.notna(row.pieces_produced) else None,
    **{col: (getattr(row, col) if pd.notna(getattr(row, col)) else None) for col in FEATURE_COLUMNS},
    **{col: getattr(row, col) for col in LABEL_COLUMNS},
) for row in df.itertuples(index=False)]

print(f"✅ {len(gold_telemetry_objects)} objets GoldTelemetry construits ({len(FEATURE_COLUMNS)} features, {len(LABEL_COLUMNS)} labels)")
for label_name in LABEL_COLUMNS:
    print(f"   {label_name} : {int(df[label_name].sum())} lignes positives")

## Nettoyage de la base de données

Suppression des données existantes dans la table `gold_telemetry` avant rechargement.

In [8]:
from sqlalchemy import delete
from models.gold import GoldTelemetry

with Session(engine) as session:
    session.execute(delete(GoldTelemetry))
    session.commit()

print("✅ Table gold_telemetry nettoyée")

✅ Table gold_telemetry nettoyée


## Insertion dans les tables gold

Insertion des objets `GoldTelemetry` construits.

In [6]:
from models.gold import GoldTelemetry

with Session(engine) as session:
    session.add_all(gold_telemetry_objects)
    session.commit()

print(f"✅ gold_telemetry : {len(gold_telemetry_objects)} lignes insérées")

✅ gold_telemetry : 400089 lignes insérées


## Dataset de référence

Téléchargement du dataset gold complet de référence (`dacodemaniak/indusense` sur Hugging Face) pour comparaison avec notre propre pipeline.

In [ ]:
import pandas as pd

reference_df = pd.read_parquet("hf://datasets/dacodemaniak/indusense/gold_dataset_20260622-080603.parquet")
reference_df.to_csv("artifacts/ingestions/gold/indusense_gold_reference.csv", index=False)

print(f"✅ Dataset de référence téléchargé et sauvegardé : {reference_df.shape[0]} lignes, {reference_df.shape[1]} colonnes")